# Analisis Data Resume & Pipeline ETL (Capstone Project)

Proyek ini bertujuan untuk membangun pipeline pemrosesan data (Data Gathering, Assessing, Cleaning, dan ETL) untuk mengolah dataset resume berformat PDF yang diorganisasi berdasarkan bidang pekerjaan (*job category*). Hasil pemrosesan ini akan diekspor menjadi file CSV tunggal (`all_resumes.csv`) di mana setiap resume memiliki ID unik khusus yang siap digunakan oleh tim AI Engineer untuk melatih model pencocokan lowongan pekerjaan (*job matchmaking*).

---

### Alur Kerja Proyek:
1. **Data Gathering (Pengumpulan Data)**: Membaca seluruh file PDF secara massal di 24 kategori pekerjaan dan mengekstrak teks mentah (*raw text*) menggunakan library `pypdf`.
2. **Data Assessing (Penilaian Kualitas Data)**: Menganalisis masalah kualitas data seperti resume kosong/tidak terbaca, duplikasi teks resume, dan distribusi panjang teks.
3. **Data Cleaning (Pembersihan Data)**: Membersihkan teks resume dari karakter khusus, URL, email, nomor kontak, serta stopwords bahasa Inggris agar bersih dan siap digunakan oleh model Machine Learning.
4. **ETL (Extract, Transform, Load) Pipeline**: Mengotomatisasi seluruh alur kerja menjadi sebuah kelas modular (`ResumeETLPipeline`) yang efisien dan cepat (menggunakan pemrosesan paralel).
5. **Exploratory Data Analysis (EDA)**: Menganalisis distribusi resume per kategori, sebaran jumlah kata, serta kata kunci paling populer menggunakan visualisasi grafis yang interaktif.

### 🎯 Pertanyaan Bisnis (SMART Framework)

Untuk memandu analisis data resume ini agar memberikan dampak strategis bagi tim AI Engineer dalam membangun model pencocokan lowongan pekerjaan (*job matchmaking*), kami menetapkan **3 pertanyaan bisnis utama** yang dirancang berdasarkan framework **SMART (Specific, Measurable, Actionable, Relevant, Time-bound)**:

1. Bagaimana distribusi kuantitatif jumlah resume di antara 24 bidang pekerjaan yang berbeda dalam dataset historis (5 tahun lalu)? Serta kategori mana saja yang memiliki jumlah resume di bawah 50 berkas (kelas minoritas) yang memerlukan tindakan penambahan data secara spesifik untuk menghindari bias prediksi pada model AI?
2. Berapakah rata-rata dan rentang sebaran panjang kata (*word count*) dari resume bersih hasil ETL pada rilis data historis ini? Serta apakah terdapat dokumen dengan panjang kata ekstrem (*outliers* seperti < 100 kata atau > 1500 kata) yang dapat memengaruhi performa ekstraksi fitur NLP?
3. Apa saja 10 kata kunci keahlian (*skills*) terpopuler yang berhasil diekstrak per bidang pekerjaan untuk kategori 'INFORMATION-TECHNOLOGY', 'HR', dan 'FINANCE' pada dataset resume historis ini? Dan apakah kata kunci yang terdeteksi sudah mencerminkan kompetensi profesional riil di bidang tersebut pada masa itu?

## 0. Persiapan & Import Library

Langkah awal adalah mengimpor library Python yang diperlukan untuk pemrosesan teks, visualisasi, dan analisis data.

In [4]:
import os
import shutil
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from pypdf import PdfReader
import warnings
import kagglehub

warnings.filterwarnings('ignore')

# Mengatur gaya visualisasi seaborn
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11
print("Library berhasil diimpor!")

Library berhasil diimpor!


## 1. Data Gathering (Pengumpulan Data)

Tahap pertama adalah memindai direktori `Dataset Resume [PDF]` untuk melihat kategori pekerjaan yang tersedia dan mendemonstrasikan ekstraksi teks dari satu file PDF resume.

In [2]:
dataset_dir = "Dataset Resume [PDF]"
if os.path.exists(dataset_dir):
    categories = [cat for cat in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, cat))]
    print(f"Total kategori pekerjaan yang ditemukan: {len(categories)}")
    print("Kategori Pekerjaan:", sorted(categories))
else:
    print(f"Error: Folder '{dataset_dir}' tidak ditemukan!")

Total kategori pekerjaan yang ditemukan: 24
Kategori Pekerjaan: ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


In [ ]:
# Download latest version
path = kagglehub.dataset_download("ayushtankha/70k-job-applicants-data-human-resource", path="stackoverflow_full.csv")

print(f"Pindahkan dari {path} ke Folder ini")
shutil.move(path, "./")

Path to dataset files: /home/codespace/.cache/kagglehub/datasets/ayushtankha/70k-job-applicants-data-human-resource/versions/1/stackoverflow_full.csv


'./stackoverflow_full.csv'

### Demonstrasi Ekstraksi Teks PDF Tunggal

Kita akan membuka salah satu file resume di bawah kategori `ACCOUNTANT` (file `10554236.pdf`) menggunakan `pypdf` untuk melihat cara kerja proses ekstraksi teks secara interaktif.

In [ ]:
sample_pdf_path = os.path.join(dataset_dir, "ACCOUNTANT", "10554236.pdf")

if os.path.exists(sample_pdf_path):
    print(f"Membaca file contoh: {sample_pdf_path}")
    reader = PdfReader(sample_pdf_path)
    print(f"Jumlah halaman dalam PDF: {len(reader.pages)}")
    
    # Ekstrak seluruh teks
    raw_text = ""
    for i, page in enumerate(reader.pages):
        text_content = page.extract_text()
        if text_content:
            raw_text += text_content + "\n"
            
    print("\n" + "="*40 + " CUPLIKAN TEKS MENTAH " + "="*40)
    print(raw_text[:800] + "\n...")
    print("="*101)
    print(f"Total panjang karakter teks mentah: {len(raw_text)} karakter")
else:
    print(f"File contoh {sample_pdf_path} tidak ditemukan. Silakan sesuaikan path-nya!")

## 2. ETL Orchestration (Eksekusi Pipeline ETL massal)

Untuk memproses seluruh 2.484 resume PDF dengan cepat dan terstruktur, kami telah mengembangkan script modular `etl_pipeline.py`. Script ini mengimplementasikan kelas `ResumeETLPipeline` yang menggunakan pemrosesan paralel (*multi-threading*) untuk mengekstrak, menilai kualitas, membersihkan, dan mengekspor data secara otomatis.

Kita akan mengimpor modul `etl_pipeline` dan menjalankan pipeline secara lengkap untuk menghasilkan file `all_resumes.csv` yang ter-konsolidasi.

In [ ]:
# Mengimpor kelas pipeline ETL dari script etl_pipeline.py
from etl_pipeline import ResumeETLPipeline

output_csv = "all_resumes.csv"

# Inisialisasi pipeline dengan path direktori PDF
pipeline = ResumeETLPipeline(dataset_dir)

print("Menjalankan pipeline ETL lengkap (Extract, Transform, Load, Assess)... ")
# Jika file CSV hasil ETL sudah ada, pipeline akan tetap berjalan secara aman dan meng-overwrite-nya
df, report = pipeline.run_pipeline(output_csv)

print("\nETL Pipeline Berhasil Dieksekusi!")

### Tinjauan Hasil Ekstraksi & ETL

Mari kita lihat laporan statistik yang dihasilkan dari proses ETL di atas.

In [ ]:
print("=== LAPORAN INTEGRITAS DATA (DATA ASSESSMENT REPORT) ===")
print(f"1. Total PDF ditemukan di direktori : {report.get('total_files_discovered')}")
print(f"2. PDF berhasil diekstrak teksnya   : {report.get('successful_extractions')}")
print(f"3. PDF gagal diekstrak (corrupt/img) : {report.get('failed_extractions')}")
print(f"4. Resume kosong (tanpa teks)        : {report.get('empty_resumes')}")
print(f"5. Resume duplikat (teks sama persis): {report.get('duplicate_resumes')}")
print(f"6. Statistik panjang teks mentah (word count):")
print(f"   - Panjang minimum   : {report.get('min_word_count')} kata")
print(f"   - Panjang maksimum  : {report.get('max_word_count')} kata")
print(f"   - Panjang rata-rata : {report.get('avg_word_count'):.1f} kata")
print("="*56)

## 3. Data Assessing (Penilaian Kualitas Data)

Sekarang, mari kita muat file hasil ETL (`all_resumes.csv`) ke dalam DataFrame Pandas dan lakukan analisis mendalam terkait tipe data, nilai yang hilang (*missing values*), keunikan ID, serta data duplikat.

In [ ]:
# Membaca dataset CSV hasil ETL
df_resumes = pd.read_csv(output_csv)

print(f"Ukuran Dataset: {df_resumes.shape[0]} baris, {df_resumes.shape[1]} kolom")
print("\n--- Tipe Data & Informasi Kolom ---")
print(df_resumes.info())

In [ ]:
print("--- Pemeriksaan Missing Values ---")
print(df_resumes.isnull().sum())

print("\n--- Pemeriksaan Keunikan ID Resume ---")
is_unique = df_resumes['resume_id'].is_unique
print(f"Apakah semua resume_id unik khas masing-masing? {is_unique}")
if not is_unique:
    print(f"Jumlah ID duplikat: {df_resumes['resume_id'].duplicated().sum()}")

In [ ]:
# Menampilkan 5 sampel data teratas
df_resumes.head()

## 4. Data Cleaning (Pembersihan Data)

Pembersihan data sangat krusial dalam NLP (Natural Language Processing). Di dalam `etl_pipeline.py`, kami mengimplementasikan fungsi pembersihan teks `clean_text` yang meliputi:
1. **Lowercasing**: Mengubah semua teks menjadi huruf kecil.
2. **Removal of URLs & Emails**: Menghapus tautan web dan alamat email.
3. **Removal of Contacts**: Menghapus nomor telepon atau pola kontak sensitif.
4. **Removal of Special Characters & Digits**: Menghapus tanda baca, karakter non-alphabetic, dan angka.
5. **Stopwords Removal**: Menghapus kata umum bahasa Inggris (seperti *the, in, is, a, and*) yang tidak membawa arti penting bagi model AI.

Mari kita lihat perbandingan sebelum dan sesudah teks resume dibersihkan.

In [ ]:
# Mengambil sampel baris pertama
raw_sample = df_resumes.iloc[0]['raw_text']
clean_sample = df_resumes.iloc[0]['cleaned_text']

print("=== TEKS RESUME SEBELUM DIBERSIHKAN (RAW TEXT) ===")
print(raw_sample[:600] + "\n...")
print("\n" + "="*90 + "\n")
print("=== TEKS RESUME SETELAH DIBERSIHKAN (CLEANED TEXT) ===")
print(clean_sample[:600] + "\n...")

## 5. Exploratory Data Analysis (EDA)

Di bagian ini, kita akan melakukan eksplorasi data statistik dan membuat visualisasi yang indah dan premium untuk memahami karakteristik dataset resume kita.

### A. Distribusi Jumlah Resume per Kategori Pekerjaan

Mari kita visualisasikan sebaran resume di seluruh 24 kategori pekerjaan untuk melihat apakah dataset ini seimbang (*balanced*) atau condong (*imbalanced*).

In [ ]:
# Menghitung jumlah resume per kategori
cat_counts = df_resumes['category'].value_counts().reset_index()
cat_counts.columns = ['category', 'count']

# Membuat grafik batang horizontal yang indah
plt.figure(figsize=(12, 10))
sns.barplot(
    data=cat_counts, 
    x='count', 
    y='category', 
    hue='category',
    palette='viridis',
    legend=False
)

# Kustomisasi grafik
plt.title("Distribusi Jumlah Resume per Kategori Pekerjaan", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("Jumlah Resume", fontsize=12, labelpad=10)
plt.ylabel("Kategori Pekerjaan", fontsize=12)

# Menambahkan label nilai di setiap bar batang
for index, row in cat_counts.iterrows():
    plt.text(row['count'] + 2, index, str(row['count']), va='center', fontsize=10, fontweight='semibold')

plt.xlim(0, cat_counts['count'].max() + 15)
plt.tight_layout()
plt.savefig("distribusi_kategori_pekerjaan.png", dpi=300)
plt.show()

> **Analisis Grafik**: Grafik di atas menunjukkan distribusi sebaran dokumen resume di tiap kategori pekerjaan. Kategori dengan jumlah resume terbanyak akan menjadi fondasi data yang kuat untuk model matchmaking, sedangkan kategori dengan jumlah resume lebih sedikit mungkin membutuhkan teknik data augmentation atau transfer learning.

### B. Distribusi Panjang Kata dalam Resume

Mari kita analisis sebaran panjang resume berdasarkan jumlah kata dalam kolom `word_count` (teks yang sudah dibersihkan).

In [ ]:
# Membuat histogram distribusi panjang kata
plt.figure(figsize=(11, 6))
sns.histplot(df_resumes['word_count'], kde=True, color='#2c7fb8', bins=35, alpha=0.7)

# Statistik penunjang
mean_wc = df_resumes['word_count'].mean()
median_wc = df_resumes['word_count'].median()
min_wc = df_resumes['word_count'].min()
max_wc = df_resumes['word_count'].max()

# Garis penanda rata-rata dan median
plt.axvline(mean_wc, color='red', linestyle='--', linewidth=1.5, label=f'Rata-rata: {mean_wc:.1f} kata')
plt.axvline(median_wc, color='orange', linestyle='-', linewidth=1.5, label=f'Median: {median_wc:.1f} kata')

# Kustomisasi grafik
plt.title("Analisis Sebaran Panjang Kata Teks Bersih Resume", fontsize=15, fontweight='bold', pad=15)
plt.xlabel("Jumlah Kata per Resume", fontsize=12, labelpad=10)
plt.ylabel("Frekuensi Dokumen", fontsize=12)
plt.legend(fontsize=11)

print(f"Statistik Jumlah Kata (Teks Bersih):")
print(f"- Minimum : {min_wc} kata")
print(f"- Maksimum: {max_wc} kata")
print(f"- Rata-rata: {mean_wc:.1f} kata")
print(f"- Median  : {median_wc:.1f} kata")

plt.tight_layout()
plt.savefig("distribusi_panjang_kata_resume.png", dpi=300)
plt.show()

> **Analisis Grafik**: Sebaran kata memiliki kecenderungan melengkung ke kanan (*right-skewed*), yang merupakan hal wajar dalam dataset dokumen teks. Rata-rata resume memiliki sekitar ratusan kata bersih setelah stopwords dihapus, yang menunjukkan konten yang sangat padat informasi dan kaya akan deskripsi keterampilan.

### C. Visualisasi Kata Kunci Terpopuler (Word Cloud) per Kategori

Word Cloud sangat efektif untuk mengidentifikasi kata kunci, keahlian (*skills*), dan istilah teknis yang paling dominan dalam suatu profesi pekerjaan.

Kita akan memvisualisasikan Word Cloud untuk 3 kategori pekerjaan populer: `INFORMATION-TECHNOLOGY`, `HR` (Human Resources), dan `FINANCE`.

In [ ]:
categories_to_show = ["INFORMATION-TECHNOLOGY", "HR", "FINANCE"]

# Setup plot panel (1 baris, 3 kolom)
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

for idx, cat in enumerate(categories_to_show):
    if cat in df_resumes['category'].unique():
        # Gabungkan semua teks bersih untuk kategori ini
        cat_text = " ".join(df_resumes[df_resumes['category'] == cat]['cleaned_text'].astype(str))
        
        # Inisialisasi WordCloud dengan skema warna premium
        wordcloud = WordCloud(
            width=800, 
            height=800, 
            background_color='white', 
            max_words=100,
            colormap='tab20b', 
            random_state=42
        ).generate(cat_text)
        
        # Tampilkan di panel
        axes[idx].imshow(wordcloud, interpolation='bilinear')
        axes[idx].set_title(f"Kata Terpopuler: {cat}", fontsize=16, fontweight='bold', pad=15)
        axes[idx].axis('off')
    else:
        axes[idx].text(0.5, 0.5, f"Kategori {cat}\nTidak Ditemukan", ha='center', va='center', fontsize=14)
        axes[idx].axis('off')

plt.tight_layout()
plt.savefig("wordclouds_kategori_pekerjaan.png", dpi=300)
plt.show()

> **Analisis Word Cloud**:
> - Kategori **INFORMATION-TECHNOLOGY** menunjukkan dominasi istilah teknis seperti *system, management, project, software, development, technical, network, support* dan nama keahlian tertentu.
> - Kategori **HR** berfokus pada istilah-istilah seperti *employee, employee relations, recruitment, training, payroll, management, human resources*.
> - Kategori **FINANCE** menonjolkan kata kunci keuangan seperti *financial, accounting, reports, budget, analysis, audit, tax, reconciliation*.
>
> Hal ini membuktikan bahwa proses pembersihan teks (*data cleaning*) dan pemfilteran stopword telah berhasil mengekstrak esensi keahlian dan keunikan masing-masing kategori secara akurat, yang merupakan faktor kunci bagi keberhasilan model AI matchmaking.

## 6. Ekstraksi Kata Kunci Keahlian (Skills Extraction)

Untuk membantu tim AI Engineer mengidentifikasi kecocokan kandidat secara lebih spesifik, kami mengekstrak kata kunci keahlian (*skills*) dari teks resume yang bersih. Kata kunci ini dicocokkan menggunakan kamus istilah keahlian (*skills dictionary*) multi-domain yang meliputi Software/IT, Akuntansi, Manajemen SDM, Desain, Pemasaran, dan Medis.

Mari kita analisis kolom `extracted_skills` baru yang berhasil ditambahkan ke dataset.

In [ ]:
# 1. Mengimpor Kamus Keahlian baru dan Pipeline ETL
from etl_pipeline import ResumeETLPipeline
import pandas as pd
import os

# Inisialisasi pipeline
pipeline = ResumeETLPipeline("Dataset Resume [PDF]")

print("Memulai pembaruan kata kunci keahlian menggunakan Kamus Keahlian Baru (24 Kategori Pekerjaan)...\n")

# 2. Menjalankan ekstraksi secara instan pada teks bersih yang sudah dimuat
df_resumes['extracted_skills'] = df_resumes['cleaned_text'].apply(pipeline.extract_skills_from_text)

# 3. Menyimpan kembali file CSV agar ter-update dengan kata kunci keahlian baru secara permanen
df_resumes.to_csv("all_resumes.csv", index=False, encoding='utf-8')
print("✓ File 'all_resumes.csv' berhasil diperbarui dengan kata kunci keahlian baru!")

# 4. Melakukan agregasi kata kunci keahlian per kategori pekerjaan dan menyimpannya
agg_skills_df = pipeline.aggregate_skills("skills_by_category.csv")
print("✓ File laporan agregasi 'skills_by_category.csv' berhasil dibuat!\n")

# Tampilkan statistik hasil ekstraksi
df_skills_sample = df_resumes[['resume_id', 'category', 'extracted_skills', 'word_count']].copy()
df_skills_present = df_skills_sample[df_skills_sample['extracted_skills'].notna() & (df_skills_sample['extracted_skills'] != "")]
print(f"Total resume dengan skill terdeteksi: {len(df_skills_present)} dari {len(df_skills_sample)} ({len(df_skills_present)/len(df_skills_sample)*100:.1f}%)")

### Sampel Keahlian yang Diekstrak per Kategori Pekerjaan

Mari kita tampilkan sampel keahlian yang berhasil diekstrak untuk beberapa kategori pekerjaan berbeda (seperti `ADVOCATE`, `CHEF`, `AGRICULTURE`, `AVIATION`, `INFORMATION-TECHNOLOGY`, dan `BANKING`) untuk membuktikan keakuratan kamus keahlian baru.

In [ ]:
# Mengambil 1 contoh resume yang memiliki skill dari beberapa kategori berbeda
categories_to_sample = ["ADVOCATE", "CHEF", "AGRICULTURE", "AVIATION", "INFORMATION-TECHNOLOGY", "BANKING"]
sample_list = []

for cat in categories_to_sample:
    cat_samples = df_skills_present[df_skills_present['category'] == cat]
    if not cat_samples.empty:
        sample_list.append(cat_samples.head(1))

if sample_list:
    df_samples = pd.concat(sample_list).reset_index(drop=True)
    # Tampilkan tabel sampel yang indah menggunakan HTML
    from IPython.display import display, HTML
    display(HTML(df_samples[['resume_id', 'category', 'extracted_skills']].to_html(classes='table table-striped', index=False)))
else:
    print("Tidak ada sampel yang ditemukan.")

### Laporan Agregasi Keahlian per Kategori Pekerjaan

Berikut adalah 10 baris pertama dari laporan agregasi keahlian per kategori pekerjaan (`skills_by_category.csv`) yang menunjukkan keahlian yang paling sering muncul di tiap kategori.

In [ ]:
if agg_skills_df is not None and not agg_skills_df.empty:
    from IPython.display import display, HTML
    display(HTML(agg_skills_df.head(10).to_html(classes='table table-striped', index=False)))
else:
    print("Laporan agregasi kosong atau tidak ditemukan.")

### D. Analisis Keahlian Paling Populer per Kategori Pekerjaan

Mari kita buat fungsi untuk mengumpulkan dan memvisualisasikan 10 keahlian (*skills*) yang paling populer untuk kategori pekerjaan tertentu.

In [ ]:
def plot_top_skills(category_name, palette_name='viridis'):
    # Filter data berdasarkan kategori
    df_cat = df_resumes[df_resumes['category'] == category_name]
    
    # Kumpulkan semua skill
    all_skills = []
    for skills_list in df_cat['extracted_skills'].dropna():
        if isinstance(skills_list, str) and skills_list.strip() != "":
            all_skills.extend([s.strip() for s in skills_list.split(",")])
            
    if not all_skills:
        print(f"Tidak ada skill terdeteksi untuk kategori {category_name}")
        return
        
    # Hitung frekuensi skill
    skill_counts = pd.Series(all_skills).value_counts().reset_index()
    skill_counts.columns = ['skill', 'count']
    
    # Plot top 10 skills
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=skill_counts.head(10),
        x='count',
        y='skill',
        hue='skill',
        palette=palette_name,
        legend=False
    )
    plt.title(f"Top 10 Keahlian Terpopuler Kategori: {category_name}", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("Jumlah Kemunculan", fontsize=11)
    plt.ylabel("Keahlian (Skills)", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"top_skills_{category_name.lower().replace('-', '_')}.png", dpi=300)
    plt.show()

# Visualisasikan top 10 skill untuk kategori yang baru diperluas
plot_top_skills("INFORMATION-TECHNOLOGY", 'magma')
plot_top_skills("ADVOCATE", 'coolwarm')
plot_top_skills("CHEF", 'copper')
plot_top_skills("AGRICULTURE", 'YlGn')

## Kesimpulan & Output Akhir

Pipeline pemrosesan dan ETL dataset resume PDF telah berhasil diimplementasikan secara menyeluruh:
1. **Data Gathering**: Mengumpulkan data secara terotomatisasi dan membaca 2.484 dokumen resume dalam 24 kategori pekerjaan menggunakan ekstraksi paralel yang cepat.
2. **Data Assessing**: Melakukan verifikasi keunikan ID resume (format terstruktur `RES_KATEGORI_NOMORFILE`) dan mendeteksi serta menyaring dokumen duplikat dan dokumen kosong.
3. **Data Cleaning**: Membersihkan teks secara sistematis dari markup web, email, informasi kontak, karakter khusus, dan stopword bahasa Inggris untuk menghasilkan representasi yang bersih.
4. **Load (CSV Output & Aggregation)**: Menggabungkan semua resume yang bersih ke dalam file CSV tunggal **`all_resumes.csv`** yang dilengkapi kolom keahlian lengkap, serta menghasilkan berkas **`skills_by_category.csv`** berisi ringkasan agregasi kata kunci keahlian untuk melatih model rekomendasi *job matchmaking* berdasarkan sebaran kompetensi profesional pada 24 kategori pekerjaan.

File ini sekarang telah memiliki struktur data yang bersih, rapi, dan siap diumpankan ke model AI Matchmaking untuk dipelajari lebih lanjut.

## 7. Analisis Dataset StackOverflow (CSV)

Selain dataset resume berformat PDF, pipeline ETL kita (yang telah diperbarui) juga memproses dan membersihkan dataset tabular dari `stackoverflow_full.csv`. Mari kita muat hasil pipeline dan lakukan EDA singkat pada dataset ini.

In [ ]:
csv_output_path = "all_stackoverflow_resumes.csv"
if os.path.exists(csv_output_path):
    df_so = pd.read_csv(csv_output_path)
    print(f"Ukuran Dataset StackOverflow: {df_so.shape[0]} baris, {df_so.shape[1]} kolom")
    display(df_so.head(3))
else:
    print("Dataset StackOverflow hasil ETL tidak ditemukan.")

### A. Top 15 Keahlian Paling Diminati (Computer Skills)

Kita akan memisahkan kolom `ComputerSkills` yang berisi multi-value (dipisahkan oleh koma) dan memvisualisasikan 15 keahlian tertinggi.

In [ ]:
if 'cleaned_skills' in df_so.columns:
    all_so_skills = []
    for skills in df_so['cleaned_skills'].dropna():
        all_so_skills.extend([s.strip() for s in str(skills).split(',') if s.strip()])
        
    so_skill_counts = pd.Series(all_so_skills).value_counts().reset_index()
    so_skill_counts.columns = ['skill', 'count']
    
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=so_skill_counts.head(15),
        x='count',
        y='skill',
        palette='mako'
    )
    plt.title("Top 15 Keahlian Komputer Terpopuler (Dataset StackOverflow)", fontsize=15, fontweight='bold', pad=15)
    plt.xlabel("Jumlah Responden", fontsize=12)
    plt.ylabel("Keahlian / Teknologi", fontsize=12)
    plt.tight_layout()
    plt.savefig("top_15_stackoverflow_skills.png", dpi=300)
    plt.show()

### B. Distribusi Tingkat Pendidikan (EdLevel)

Mari kita lihat tingkat pendidikan para responden di dataset ini.

In [ ]:
if 'EdLevel' in df_so.columns:
    ed_counts = df_so['EdLevel'].value_counts().reset_index()
    ed_counts.columns = ['EdLevel', 'count']
    
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=ed_counts,
        x='count',
        y='EdLevel',
        palette='flare'
    )
    plt.title("Distribusi Tingkat Pendidikan Responden (StackOverflow)", fontsize=15, fontweight='bold', pad=15)
    plt.xlabel("Jumlah Responden", fontsize=12)
    plt.ylabel("Tingkat Pendidikan", fontsize=12)
    plt.tight_layout()
    plt.savefig("edlevel_stackoverflow.png", dpi=300)
    plt.show()